In [ ]:
import os
import json
import numpy as np
from datasets import load_dataset, Dataset, load_from_disk
import torch
import wandb
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

from trl import SFTTrainer, SFTConfig

In [ ]:
TOKENIZER_ID = "Qwen/Qwen3-1.7B"
MODEL_ID     = "Qwen/Qwen3-1.7B"
DATASET_DIR  = "/ultrachat_5m"
OUTPUT_DIR   = "/pipeline_b"
TOKEN_BUDGET = 5000000
MAX_SEQ_LEN  = 2048
SEED         = 42
 
lora_rank    = 16
lora_alpha   = 32
lora_dropout = 0.05
epochs       = 1
lr           = 2e-4
batch_size   = 4
grad_accum   = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def format_messages(tokenizer, example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

def get_dataset():
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
 
    print("Loading UltraChat-200k (streaming)...")
    ds = load_dataset(
        "HuggingFaceH4/ultrachat_200k",
        split="train_sft",
        streaming=True,
    )
    ds = ds.shuffle(seed=SEED, buffer_size=20_000)
 
    print(f"Subsampling to {TOKEN_BUDGET:,} token budget (max_seq_len={MAX_SEQ_LEN})...")
 
    kept_examples = []
    total_tokens  = 0
    dropped_over_limit = 0
    processed = 0
 
    for example in ds:
        text = format_messages(tokenizer, example)
        ids  = tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
            return_attention_mask=False,
        )["input_ids"]
 
        length = len(ids)
        processed += 1

        if length > MAX_SEQ_LEN:
            dropped_over_limit += 1
            continue
 
        if total_tokens + length > TOKEN_BUDGET:
            break
 
        kept_examples.append({
            "text":          text,
            "messages":      example["messages"],
            "token_length":  length,
        })
        total_tokens += length
 
        if len(kept_examples) % 500 == 0:
            print(f"  Kept {len(kept_examples):,} examples | "
                  f"{total_tokens:,} / {TOKEN_BUDGET:,} tokens")
 
    lengths = np.array([ex["token_length"] for ex in kept_examples])
    print(f"\n{'─'*50}")
    print("  Final dataset summary")
    print(f"{'─'*50}")
    print(f"  Examples kept     : {len(kept_examples):,}")
    print(f"  Total tokens      : {total_tokens:,}")
    print(f"  Dropped (>{MAX_SEQ_LEN}): {dropped_over_limit:,}")
    print(f"  Processed total   : {processed:,}")
    print(f"  Mean length       : {lengths.mean():.1f}")
    print(f"  Median length     : {np.median(lengths):.1f}")
    print(f"  Max length        : {lengths.max()}")
    print(f"{'─'*50}")
 
    os.makedirs(DATASET_DIR, exist_ok=True)
 
    hf_dataset = Dataset.from_list(kept_examples)
    hf_dataset.save_to_disk(DATASET_DIR)
 
    meta = {
        "tokenizer":       TOKENIZER_ID,
        "token_budget":    TOKEN_BUDGET,
        "max_seq_len":     MAX_SEQ_LEN,
        "seed":            SEED,
        "examples_kept":   len(kept_examples),
        "total_tokens":    total_tokens,
        "mean_length":     float(lengths.mean()),
        "median_length":   float(np.median(lengths)),
        "max_length":      int(lengths.max()),
        "dropped_over_limit": dropped_over_limit,
    }
    with open(os.path.join(DATASET_DIR, "metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)
 
    print(f"\n  Saved to {DATASET_DIR}")
    print(f"  Load with: Dataset.load_from_disk('{DATASET_DIR}')")

In [ ]:
get_dataset()

In [ ]:
ds = Dataset.load_from_disk(DATASET_DIR)
text = ds[0]["text"]
print(ds)
print(ds[0]["text"][:500])
print(repr(text[-100:]))

In [ ]:
def train_qlora():
    wandb.init(
        project="ptq-vs-qlora",
        name="qwen3-1.7b_b_ultrachat_s0",
        config={
            "model":        MODEL_ID,
            "dataset":      "ultrachat_5m",
            "max_seq_len":  MAX_SEQ_LEN,
            "lora_rank":    lora_rank,
            "lora_alpha":   lora_alpha,
            "lora_dropout": lora_dropout,
            "epochs":       epochs,
            "lr":           lr,
            "batch_size":   batch_size,
            "grad_accum":   grad_accum,
            "seed":         SEED,
            "pipeline":     "B",
            "quant_type":   "nf4",
        },
    )
     
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
     
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
     
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        dtype=torch.float16,
        device_map=DEVICE,
    )
    
    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False
    model.config.pad_token_id = tokenizer.pad_token_id
          
    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(
        model,
        lora_config,
        autocast_adapter_dtype=False,
    )    
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.print_trainable_parameters()

     
    args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        gradient_checkpointing=True,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_steps=100,
        fp16=True,
        optim="paged_adamw_8bit",
        logging_steps=10,
        save_steps=200,
        save_total_limit=3,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
    )
     
    trainer = SFTTrainer(
        model=model,
        args=args,
        processing_class=tokenizer,
        train_dataset=load_from_disk(DATASET_DIR),
    )
        
    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    print(f"Pipeline B saved to {OUTPUT_DIR}")    

In [ ]:
train_qlora()

In [ ]:
def model_inference():
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
    model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-1.7B",
        quantization_config=bnb_config,
        device_map=DEVICE,
    )
    model = PeftModel.from_pretrained(model, OUTPUT_DIR)

    messages = [
        {"role": "user", "content": "how many r in strawberry"}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
    )
    print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
import string
from tqdm import tqdm
 
PIPELINE        = "B"
SEED            = 42
OUTPUT_DIR      = "/output"
PIPELINE_B_BASE = "Qwen/Qwen3-1.7B"
PIPELINE_B_ADAPTER = "/pipeline_b"
SAVE_EVERY      = 200 

LETTERS = list(string.ascii_uppercase)

In [ ]:
def load_model(pipeline):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(PIPELINE_B_ADAPTER)
    model = AutoModelForCausalLM.from_pretrained(
        PIPELINE_B_BASE,
        quantization_config=bnb_config,
        dtype=torch.float16,
        device_map=DEVICE,
    )
    model = PeftModel.from_pretrained(model, PIPELINE_B_ADAPTER)
    model.eval()
    return model, tokenizer

    
def get_letter_ids(tokenizer, n):
    assert n <= len(LETTERS)
    return [
        tokenizer.encode(l, add_special_tokens=False)[0]
        for l in LETTERS[:n]
    ]
 
 
def format_prompt(tokenizer, question, choices):
    choice_str = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(choices))
    content = (
        f"The following is a multiple choice question. "
        f"Answer with only a single letter.\n\n"
        f"{question}\n{choice_str}"
    )
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=False,
        add_generation_prompt=True,
    )


In [ ]:
@torch.no_grad()
def get_logits(model, tokenizer, prompt, letter_ids):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model(**inputs)
    last = out.logits[0, -1, :]
    return torch.stack([last[i] for i in letter_ids]).float().cpu().numpy()
 
 
def load_mmlu():
    ds = load_dataset("cais/mmlu", "all", split="test")
    return [
        {
            "benchmark": "mmlu",
            "subject": ex["subject"],
            "question": ex["question"],
            "choices": ex["choices"],
            "true_label": ex["answer"],
        }
        for ex in ds
    ]
 
 
def load_arc():
    ds = load_dataset("ai2_arc", "ARC-Challenge", split="test")
    label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "1": 0, "2": 1, "3": 2, "4": 3}
    examples = []
    for ex in ds:
        choices = ex["choices"]["text"]
        true_label = label_map.get(ex["answerKey"], -1)
        if len(choices) != 4 or true_label == -1:
            continue
        examples.append({
            "benchmark": "arc_challenge",
            "subject": "arc_challenge",
            "question": ex["question"],
            "choices": choices,
            "true_label": true_label,
        })
    return examples
 
 
def load_truthfulqa():
    ds = load_dataset("truthful_qa", "multiple_choice", split="validation")
    examples = []
    for ex in ds:
        choices = ex["mc1_targets"]["choices"]
        labels  = ex["mc1_targets"]["labels"]
        if 1 not in labels:
            continue
        examples.append({
            "benchmark": "truthfulqa",
            "subject": "truthfulqa",
            "question": ex["question"],
            "choices": choices,
            "true_label": labels.index(1),
        })
    return examples
 
 
def evaluate():
    torch.manual_seed(SEED)
    out_path = os.path.join(OUTPUT_DIR, f"eval_results_{PIPELINE.lower()}.json")
 
    print(f"Loading Pipeline {PIPELINE}...")
    model, tokenizer = load_model(PIPELINE)
    print(f"GPU memory after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")
 
    examples = load_mmlu() + load_arc() + load_truthfulqa()
    print(f"Total examples: {len(examples):,}")
 
    results = []
    for i, ex in enumerate(tqdm(examples)):
        n = len(ex["choices"])
        letter_ids = get_letter_ids(tokenizer, n)
        prompt = format_prompt(tokenizer, ex["question"], ex["choices"])
        logits = get_logits(model, tokenizer, prompt, letter_ids)
        pred = int(np.argmax(logits))
        results.append({
            "benchmark":  ex["benchmark"],
            "subject":    ex["subject"],
            "true_label": ex["true_label"],
            "logits":     logits.tolist(),
            "pred_label": pred,
            "correct":    pred == ex["true_label"],
        })
        if (i + 1) % SAVE_EVERY == 0:
            with open(out_path, "w") as f:
                json.dump({"pipeline": PIPELINE, "results": results}, f)
 
    with open(out_path, "w") as f:
        json.dump({"pipeline": PIPELINE, "results": results}, f)
    print(f"Saved to {out_path}")
 
    for bench in ["mmlu", "arc_challenge", "truthfulqa"]:
        br = [r for r in results if r["benchmark"] == bench]
        acc = np.mean([r["correct"] for r in br])
        print(f"{bench}: {acc:.4f} ({len(br)} examples)")

In [ ]:
evaluate()

In [ ]:
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

RESULTS_B = "eval_results_b.json"
N_BINS    = 15

In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()


def load_results(path):
    with open(path) as f:
        data = json.load(f)
    return data["results"]

In [ ]:
def get_probs_and_correct(results):
    probs = []
    correct = []
    confidences = []
    for r in results:
        p = softmax(np.array(r["logits"]))
        probs.append(p)
        correct.append(int(r["correct"]))
        confidences.append(float(np.max(p)))
    return np.array(confidences), np.array(correct), probs


def compute_ece(confidences, correct, n_bins=N_BINS):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_stats = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            bin_stats.append((lo, hi, 0, 0.0, 0.0))
            continue
        bin_acc = correct[mask].mean()
        bin_conf = confidences[mask].mean()
        weight = mask.sum() / len(confidences)
        ece += weight * abs(bin_acc - bin_conf)
        bin_stats.append((lo, hi, mask.sum(), bin_acc, bin_conf))
    return ece, bin_stats


def compute_brier(probs, true_labels, n_classes_per_example):
    scores = []
    for p, t in zip(probs, true_labels):
        onehot = np.zeros(len(p))
        onehot[t] = 1.0
        scores.append(np.sum((p - onehot) ** 2))
    return float(np.mean(scores))


def compute_nll(probs, true_labels):
    eps = 1e-12
    scores = []
    for p, t in zip(probs, true_labels):
        scores.append(-np.log(max(p[t], eps)))
    return float(np.mean(scores))


def compute_overconfidence_rate(confidences, correct, threshold=0.7):
    mask = confidences >= threshold
    if mask.sum() == 0:
        return 0.0
    return float((1 - correct[mask]).mean())


def temperature_scale(logits_list, true_labels, n_classes_per_example):
    def nll_at_temp(T):
        total = 0.0
        for logits, t in zip(logits_list, true_labels):
            scaled = np.array(logits) / T
            p = softmax(scaled)
            total += -np.log(max(p[t], 1e-12))
        return total / len(logits_list)

    res = minimize_scalar(nll_at_temp, bounds=(0.05, 10.0), method="bounded")
    return res.x, res.fun

In [ ]:
def analyze_pipeline(results, label="B"):
    confidences, correct, probs = get_probs_and_correct(results)
    true_labels = [r["true_label"] for r in results]
    logits_list = [r["logits"] for r in results]

    ece, bin_stats = compute_ece(confidences, correct)
    brier = compute_brier(probs, true_labels, None)
    nll = compute_nll(probs, true_labels)
    overconf = compute_overconfidence_rate(confidences, correct)
    accuracy = correct.mean()

    temp, temp_nll = temperature_scale(logits_list, true_labels, None)
    scaled_probs = [softmax(np.array(l) / temp) for l in logits_list]
    scaled_conf = np.array([np.max(p) for p in scaled_probs])
    ece_scaled, _ = compute_ece(scaled_conf, correct)

    print(f"\n{'='*55}")
    print(f"Pipeline {label}")
    print(f"{'='*55}")
    print(f"Accuracy            : {accuracy:.4f}")
    print(f"ECE                 : {ece:.4f}")
    print(f"ECE (temp scaled)   : {ece_scaled:.4f}")
    print(f"Brier Score         : {brier:.4f}")
    print(f"NLL                 : {nll:.4f}")
    print(f"Overconfidence rate : {overconf:.4f} (conf>=0.7, wrong)")
    print(f"Optimal temperature : {temp:.3f}")
    print(f"{'='*55}")

    subjects = sorted(set(r["subject"] for r in results if r["benchmark"] == "mmlu"))
    print("\nPer-subject MMLU ECE (top 10 by example count):")

    subject_ece = []
    for subj in subjects:
        subj_results = [r for r in results if r["subject"] == subj]
        if len(subj_results) < 5:
            continue

        sc, scor, _ = get_probs_and_correct(subj_results)
        s_ece, _ = compute_ece(sc, scor)

        subject_ece.append((subj, len(subj_results), s_ece, scor.mean()))

    subject_ece.sort(key=lambda x: -x[1])

    for subj, n, s_ece, s_acc in subject_ece[:10]:
        print(f"{subj:<35} n={n:<5} acc={s_acc:.3f}  ece={s_ece:.4f}")

    return {
        "label": label,
        "accuracy": accuracy,
        "ece": ece,
        "ece_scaled": ece_scaled,
        "brier": brier,
        "nll": nll,
        "overconf": overconf,
        "temperature": temp,
        "bin_stats": bin_stats,
        "subject_ece": subject_ece,
        "confidences": confidences,
        "correct": correct,
    }


In [ ]:
def plot_reliability_diagram(stats, save_path="reliability_diagram.png"):
    plt.figure(figsize=(6, 5.5))

    bin_stats = stats["bin_stats"]

    centers = [(lo + hi) / 2 for lo, hi, n, acc, conf in bin_stats if n > 0]
    accs = [acc for lo, hi, n, acc, conf in bin_stats if n > 0]

    plt.bar(
        centers,
        accs,
        width=1.0 / N_BINS,
        edgecolor="black",
        alpha=0.85,
    )

    plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")

    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(f"Pipeline {stats['label']} (ECE={stats['ece']:.4f})")
    plt.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Reliability diagram saved to {save_path}")
    plt.show()

In [ ]:
def evaluation_analysis():
    results = load_results(RESULTS_B)

    stats = analyze_pipeline(results, "B")

    print(f"\n{'='*55}")
    print("Summary")
    print(f"{'='*55}")

    metrics = [
        ("accuracy", "Accuracy"),
        ("ece", "ECE"),
        ("ece_scaled", "ECE (scaled)"),
        ("brier", "Brier"),
        ("nll", "NLL"),
        ("overconf", "Overconf. rate"),
        ("temperature", "Temperature"),
    ]

    for key, name in metrics:
        print(f"{name:<22}: {stats[key]:.4f}")

    print(f"{'='*55}")

    plot_reliability_diagram(stats)

    summary = {
        "pipeline": {
            "label": stats["label"],
            "accuracy": float(stats["accuracy"]),
            "ece": float(stats["ece"]),
            "ece_scaled": float(stats["ece_scaled"]),
            "brier": float(stats["brier"]),
            "nll": float(stats["nll"]),
            "overconf": float(stats["overconf"]),
            "temperature": float(stats["temperature"]),

            "bin_stats": [
                {
                    "bin_start": float(lo),
                    "bin_end": float(hi),
                    "count": int(n),
                    "accuracy": float(acc),
                    "confidence": float(conf),
                }
                for lo, hi, n, acc, conf in stats["bin_stats"]
            ],

            "subject_ece": [
                {
                    "subject": subj,
                    "n_examples": int(n),
                    "ece": float(ece),
                    "accuracy": float(acc),
                }
                for subj, n, ece, acc in stats["subject_ece"]
            ],

            "confidences": stats["confidences"].tolist(),
            "correct": stats["correct"].astype(int).tolist(),
        }
    }

    with open("calibration_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nSummary saved to calibration_summary.json")


In [ ]:
evaluation_analysis()